# Run the dynamic model for multiple locations and seasons

In [1]:
import os
import gc
import importlib
from copy import deepcopy
from pathlib import Path


import epiweeks
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

from inframind_proteus.outbreak_dynamics import RenewalSimulator, build_calibration_params_df, build_initial_infec_df
from inframind_proteus.outbreak_dynamics.utils import (
    load_yaml_dict, apply_include_exclude_logic, map_parallel_or_sequential_chunks, save_yaml_dict
)

In [2]:
class DiseaseTimeSeriesCache:

    def __init__(
            self,
            disease_cases_dir: Path,
            fname_fmt: str = "dengue_{uf}.csv"
    ):
        self.disease_cases_dir = Path(disease_cases_dir)
        self.fname_fmt = fname_fmt
        self.cache = {}

    def get(self, uf: str) -> pd.Series:
        if uf not in self.cache:
            cases_df = pd.read_csv(
                self.disease_cases_dir / self.fname_fmt.format(uf=uf),
                parse_dates=["date"],
            )
            cases_sr = cases_df.set_index("date")["value"]
            self.cache[uf] = cases_sr
        return self.cache[uf]



In [3]:
# Parameters of the multi-location code
# ================

base_config_path = Path("../../configs/prototype_run_dynamic_model.yaml")
base_config_dict = load_yaml_dict(Path("../../configs/prototype_run_dynamic_model.yaml"))
# cases_cache = DiseaseTimeSeriesCache(Path("../../data/disease/dengue_cases_uf_weekly"))

zero_date_epiweek = 41
sim_start_epiweek = 26  # Using zero_date minus at least 2x the maximum generation time in weeks
calibration_start_epiweek = 41
calibration_end_epiweek = 40  # Of the next year

use_location_ids = ["SP", "SE", "PI"]
exclude_location_ids = ["CE"]  # Just to test.
use_years = list(range(2018, 2024))



In [4]:

# ==========

# Load UF table to lookup locations
uf_table_df = pd.read_csv(Path("../../data/demographic/uf_table.csv"))

# Combine all years and locations to be run
location_ids = apply_include_exclude_logic(
    uf_table_df["uf"],
    include_list=use_location_ids,
    exclude_list=exclude_location_ids,
)
years = list(use_years)
location_year_index = pd.MultiIndex.from_product(
    [location_ids, years],
    names=["uf", "season"]
)


def run_simulation_for_location_year(location_year_tuple):
    location_id, year = location_year_tuple
    print(f"Iterating for {location_id} in {year}...")

    # Load again observations, since this function will run in parallel... not worth caching
    obserations_sr = DiseaseTimeSeriesCache(Path("../../data/disease/dengue_cases_uf_weekly")).get(location_id)

    # Create and modify the simulator and config object
    config_dict = deepcopy(base_config_dict)
    config_dict["location"]["location_id"] = location_id
    config_dict["simulation"]["population_size"] = uf_table_df.set_index("uf").loc[location_id, f"population_{year}"].item()
    _temporal = config_dict["temporal"]
    _temporal["zero_date"] = epiweeks.Week(year, zero_date_epiweek).startdate().isoformat()
    _temporal["sim_start"] = epiweeks.Week(year, sim_start_epiweek).startdate().isoformat()
    _temporal["calibration_start"] = epiweeks.Week(year, calibration_start_epiweek).startdate().isoformat()
    _temporal["calibration_end"] = epiweeks.Week(year + 1, calibration_end_epiweek).startdate().isoformat()

    simulator = RenewalSimulator.from_config_dict(config_dict)
    config = simulator.config

    # --- Build auxiliary data frames for simulations
    params_df = build_calibration_params_df(config.num_simulations, config.sampling)
    config.num_simulations = num_simulations = params_df.shape[0]  # Update in case sampling method changes number of simulations
    initial_infec_df = build_initial_infec_df(
        config.num_simulations, simulator._gt_max_steps, config.temporal.step_dt,
        config.initial_infections
    )

    # Run
    # ======
    results = simulator.run(
        params_df=params_df,
        initial_infec_df=initial_infec_df,
        observations_sr=obserations_sr,
    )


    # [PROTOTYPE] Select best simulations via WIS
    wis_sr = results.scoring.summary["wis"]
    # -()- By fraction of number of simulations
    _nsim = wis_sr.shape[0]
    _frac = 0.001
    _n = np.ceil(_frac * _nsim).astype(int)

    selected_wis_sr = wis_sr.nsmallest(_n)

    # [PROTOTYPE]: Export results
    # =====
    _root = Path("../../")
    out_dir = _root / Path(config_dict["output"]["main_dir"]) / "calibration_results" / f"{location_id}_{year}"
    out_dir.mkdir(exist_ok=True, parents=True)

    save_yaml_dict(config_dict, out_dir / "config.yaml")
    results.scoring.summary.to_csv(out_dir / "scoring.csv")
    # results.case_beam_df.to_csv(out_dir / "case_beam_df.csv")  # TOOOOOOOOO heavy!
    # Export only selected trajectories (case beams) to save space
    results.case_beam_df.reset_index().set_index("i_simulation").loc[selected_wis_sr.index].to_csv(out_dir / "case_beam_df_selected.csv")
    print(f"Done: {out_dir}")


map_parallel_or_sequential_chunks(
    run_simulation_for_location_year,
    location_year_index,
    ncpus=4, chunksize=1
)
# run_simulation_for_location_year(("SP", 2023))
print("Done all simulations!")


Iterating for SP in 2018...Iterating for SP in 2020...

Iterating for SP in 2022...Iterating for SE in 2018...

Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SP_2020
Iterating for SP in 2021...
Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SE_2018
Iterating for SE in 2019...
Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SP_2018
Iterating for SP in 2019...
Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SP_2022
Iterating for SP in 2023...
Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SP_2019
Iterating for SE in 2020...
Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SP_2021
Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SE_2019Iterating for SE in 2022...

Iterating for PI in 2018...
Done: ../../outputs/prototype_run_dynamic_model/calibration_results/SP_2023
Iterating for PI in 2020...
Done: ../../outputs/prototype_run_dynamic_model/calibrat

In [10]:

# Try loading one
# ----
fpath = Path("../../outputs/prototype_run_dynamic_model/calibration_results/PI_2018/case_beam_df_selected.csv")
df = pd.read_csv(fpath, index_col=["i_simulation", "quantile"])
df.columns = pd.to_datetime(df.columns)

df

2018-06-24  2018-07-01  2018-07-08  2018-07-15  \
i_simulation quantile                                                   
918          0.025     124.040951  124.040907  124.039279  124.036184   
             0.250     148.172726  148.172675  148.170804  148.167246   
             0.500     161.753290  161.753236  161.751228  161.747411   
             0.750     175.978231  175.978173  175.976023  175.971934   
             0.975     204.906710  204.906644  204.904205  204.899567   
598          0.025     124.040951  124.040907  124.039279  124.036184   
             0.250     148.172726  148.172675  148.170804  148.167246   
             0.500     161.753290  161.753236  161.751228  161.747411   
             0.750     175.978231  175.978173  175.976023  175.971934   
             0.975     204.906710  204.906644  204.904205  204.899567   

                       2018-07-22  2018-07-29  2018-08-05  2018-08-12  \
i_simulation quantile                                                   
918          0.025     124.034308  124.032442  124.030104  124.027981   
             0.250     148.165089  148.162943  148.160256  148.157814   
             0.500     161.745097  161.742794  161.739911  161.737291   
             0.750     175.969456  175.966990  175.963901  175.961096   
             0.975     204.896755  204.893957  204.890454  204.887271   
598          0.025     124.034308  124.032442  124.030104  124.027981   
             0.250     148.165089  148.162943  148.160256  148.157814   
             0.500     161.745097  161.742794  161.739911  161.737291   
             0.750     175.969456  175.966990  175.963901  175.961096   
             0.975     204.896755  204.893957  204.890454  204.887271   

                       2018-08-19  2018-08-26  ...  2019-08-04  2019-08-11  \
i_simulation quantile                          ...                           
918          0.025     124.025938  124.023777  ...   55.390414   43.332416   
             0.250     148.155466  148.152981  ...   68.957810   54.919401   
             0.500     161.734772  161.732105  ...   76.629349   61.487179   
             0.750     175.958397  175.955542  ...   84.686228   68.394218   
             0.975     204.884210  204.880970  ...  101.122074   82.506646   
598          0.025     124.025938  124.023777  ...   41.721382   32.450401   
             0.250     148.155466  148.152981  ...   53.037562   42.168051   
             0.500     161.734772  161.732105  ...   59.454596   47.696440   
             0.750     175.958397  175.955542  ...   66.204706   53.522125   
             0.975     204.884210  204.880970  ...   80.000287   65.452825   

                       2019-08-18  2019-08-25  2019-09-01  2019-09-08  \
i_simulation quantile                                                   
918          0.025      33.731769   26.105360   20.064318   15.294226   
             0.250      43.675132   34.675153   27.477930   21.727351   
             0.500      49.328861   39.566992   31.730371   25.439375   
             0.750      55.284853   44.731387   36.231522   29.380965   
             0.975      67.478246   55.330081   45.496517   37.522967   
598          0.025      25.089108   19.260808   14.661070   11.044679   
             0.250      33.469473   26.514276   20.957782   16.523098   
             0.500      38.256418   30.678323   24.594778   19.711004   
             0.750      43.311932   35.087879   28.458771   23.111044   
             0.975      53.691510   44.168898   36.445228   30.168580   

                       2019-09-15  2019-09-22  2019-09-29  2019-10-06  
i_simulation quantile                                                  
918          0.025      11.541555    8.602198    6.311717    4.537733  
             0.250      17.137011   13.476943   10.562280    8.244446  
             0.500      20.389039   16.334776   13.080164   10.467460  
             0.750      23.855264   19.394470   15.790005   12.874480  
             0.975     